<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing/blob/Causative-Factor-Identification/notebooks/CausationIdentification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import json
import joblib
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from google.colab import files

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    precision_recall_curve,
    average_precision_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

In [ ]:
# /content/drive/MyDrive/CrimePredictionSystem/CrimeData_Final.csv

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/CrimePredictionSystem/CrimeData_Final.csv')

In [ ]:
#Converts date to a proper datetime object
df['date']= pd.to_datetime(df['date'])

In [ ]:
df.head()

,crime,location,date,sex,victim_ethnicity,victim_age,time,weather,latitude,longitude,...,gn_pcode,gn_population,gn_distance_m,victim_ethnicity,DS_Division,Avg_Household_Income,Unemployment_Rate,Building_Density,Road_Density,Land_Area_Density
0,burglary,mulgampola,2019-12-31,f,muslim,54,8:17:00,Cloudy,7.280544,80.616500,...,LK2130170,21826,311.8,muslim,Gangawata Korale,124797,4.55,740.9,29.5,16.06
1,burglary,car park,2020-01-04,m,sinhala,42,2:00:00,Rainy,7.283445,80.619385,...,LK2130105,8913,352.4,sinhala,Gangawata Korale,111768,4.53,892.7,29.8,19.76
2,theft,bus stand,2020-01-08,f,sinhala,20,21:01:00,Rainy,7.256425,80.590461,...,LK2139135,16411,518.6,sinhala,Udunuwara,69786,5.57,548.4,16.6,11.66
3,vehicle theft,aniwatte,2020-01-10,m,sinhala,29,12:10:00,Cloudy,7.290058,80.622438,...,LK2130100,1107,381.5,sinhala,Gangawata Korale,107410,4.68,881.2,24.1,17.18
4,robbery,dutugamunu mawatha,2020-01-11,m,sinhala,59,2:39:00,Rainy,7.312344,80.645687,...,LK2130050,1293,322.6,sinhala,Gangawata Korale,120663,4.71,585.4,24.0,9.45


In [ ]:
print("Rows, Columns =",df.shape)


Rows, Columns = (1608, 24)


In [ ]:
df.columns.tolist()

['crime',
 'location',
 'date',
 'sex',
 'victim_ethnicity',
 'victim_age',
 'time',
 'weather',
 'latitude',
 'longitude',
 'holiday_name',
 'is_holiday',
 'land_use_type',
 'gn_division',
 'gn_pcode',
 'gn_population',
 'gn_distance_m',
 'victim_ethnicity ',
 'DS_Division',
 'Avg_Household_Income',
 'Unemployment_Rate',
 'Building_Density',
 'Road_Density',
 'Land_Area_Density']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1608 entries, 0 to 1607
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   crime                 1608 non-null   object        
 1   location              1608 non-null   object        
 2   date                  1608 non-null   datetime64[ns]
 3   sex                   1608 non-null   object        
 4   victim_ethnicity      1608 non-null   object        
 5   victim_age            1608 non-null   int64         
 6   time                  1608 non-null   object        
 7   weather               1608 non-null   object        
 8   latitude              1608 non-null   float64       
 9   longitude             1608 non-null   float64       
 10  holiday_name          1608 non-null   object        
 11  is_holiday            1608 non-null   int64         
 12  land_use_type         1608 non-null   object        
 13  gn_division       

In [ ]:
df.describe(include="all")

,crime,location,date,sex,victim_ethnicity,victim_age,time,weather,latitude,longitude,...,gn_pcode,gn_population,gn_distance_m,victim_ethnicity,DS_Division,Avg_Household_Income,Unemployment_Rate,Building_Density,Road_Density,Land_Area_Density
count,1608,1608,1608,1608,1608,1608.000000,1608,1608,1608.000000,1608.000000,...,1608,1608.000000,1608.000000,1608,1608,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000
unique,6,125,NaN,3,3,NaN,916,3,NaN,NaN,...,72,NaN,NaN,3,11,NaN,NaN,NaN,NaN,NaN
top,burglary,bus stand,NaN,m,sinhala,NaN,13:31:00,Rainy,NaN,NaN,...,LK2130145,NaN,NaN,sinhala,Gangawata Korale,NaN,NaN,NaN,NaN,NaN
freq,462,87,NaN,1171,1343,NaN,6,1361,NaN,NaN,...,258,NaN,NaN,1343,1485,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,2022-11-10 14:58:12.537313280,NaN,NaN,42.602612,NaN,NaN,7.285506,80.633042,...,NaN,8630.593284,398.776306,NaN,NaN,111520.689055,4.633800,676.903607,17.181157,13.717693
min,NaN,NaN,2019-12-31 00:00:00,NaN,NaN,15.000000,NaN,NaN,7.094806,80.528761,...,NaN,926.000000,6.700000,NaN,NaN,49472.000000,4.280000,0.000000,0.000000,0.000000
25%,NaN,NaN,2021-10-16 18:00:00,NaN,NaN,32.000000,NaN,NaN,7.279031,80.624353,...,NaN,2198.000000,220.675000,NaN,NaN,107400.250000,4.410000,424.375000,13.200000,8.997500
50%,NaN,NaN,2023-01-04 12:00:00,NaN,NaN,42.000000,NaN,NaN,7.289164,80.634393,...,NaN,6925.000000,316.150000,NaN,NaN,113834.500000,4.530000,772.800000,18.800000,15.800000
75%,NaN,NaN,2023-12-12 06:00:00,NaN,NaN,52.000000,NaN,NaN,7.294233,80.639486,...,NaN,13050.000000,445.600000,NaN,NaN,119886.000000,4.650000,883.800000,21.700000,19.770000
max,NaN,NaN,2025-09-23 00:00:00,NaN,NaN,78.000000,NaN,NaN,7.499662,80.764916,...,NaN,22268.000000,8483.300000,NaN,NaN,126467.000000,8.180000,1480.600000,30.100000,22.730000


##Identifying Unique values in each colummn

In [ ]:
categorical_cols =  ['crime',
 'location',
 'date',
 'sex',
 'victim_ethnicity',
 'victim_age',
 'time',
 'weather',
 'latitude',
 'longitude',
 'holiday_name',
 'is_holiday',
 'land_use_type',
 'gn_division',
 'gn_pcode',
 'gn_population',
 'gn_distance_m',
 'victim_ethnicity ',
 'DS_Division',
 'Avg_Household_Income',
 'Unemployment_Rate',
 'Building_Density',
 'Road_Density',
 'Land_Area_Density']


In [ ]:
print("Unique values")
for col in categorical_cols:
    if col in df.columns:
        print(f"{col} : ", df[col].unique())

print("Unique counts")
for col in categorical_cols:
    if col in df.columns:
        print(f"{col} : ", df[col].nunique())

Unique values
crime :  ['burglary' 'theft' 'vehicle theft' 'robbery' 'drugs' 'stabbing']
location :  ['mulgampola' 'car park' 'bus stand' 'aniwatte' 'dutugamunu mawatha'
 'mahamaya primary' 'bahirawakanda' 'thannekumbura' 'dangolla junction'
 'kandy city centre' 'lewella bangalaawatte' 'meekanuwa watarauma'
 'wattaranthanne ground' 'asgiriya stadium' 'gatambe junction'
 'dalada maligawa' 'yatinuwara street' 'hanthane kadurata malshalawa'
 'pattiyakale watte' 'municipal indoor sports complex'
 'pathanawatte heerassagala' 'ogastawatte' 'mahaiyawa' 'gatambe bridge'
 'dodamwela' 'buwelikada' 'kotugodalla road' 'ampitiya town' 'bowalawatte'
 'thalathuoya road' 'peradeniya road' 'heerassagala junction' 'udabowala'
 'post office' "people's bank" 'kandy hospital' 'hewaheta' 'train station'
 'galkaduwa junction' 'sri wickramasinghe mawatha' 'goerge.e.silva lane'
 'hanthane' 'william gopallawa lane' 'common market complex'
 'mada bowala angampitiya' 'bogambara' 'nuwarawela ampitiya' 'nagasthanne

In [ ]:
# Clean column names
df.columns = df.columns.str.strip()

# Check and remove duplicate column names created after stripping
duplicate_cols = df.columns[df.columns.duplicated()].tolist()
print("Duplicate columns after stripping:", duplicate_cols)

df = df.loc[:, ~df.columns.duplicated()].copy()

# Clean key string columns
string_cols = [
    'crime', 'location', 'sex', 'victim_ethnicity', 'time', 'weather',
    'holiday_name', 'is_holiday', 'land_use_type', 'gn_division',
    'gn_pcode', 'DS_Division'
]

for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

Duplicate columns after stripping: ['victim_ethnicity']


In [ ]:
numeric_cols = [
    'victim_age',
    'gn_population',
    'gn_distance_m',
    'Avg_Household_Income',
    'Unemployment_Rate',
    'Building_Density',
    'Road_Density',
    'Land_Area_Density'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

##Aggregate to : GN Division & Time Window

In [ ]:
## Aggregate to GN Division & Weekly Time Window

df['week'] = df['date'].dt.to_period('W').apply(lambda r: r.start_time)

aggregated_df = (
    df.groupby(['gn_division', 'week'])
      .agg(
          total_crimes=('crime', 'count'),
          burglary_count=('crime', lambda x: x.astype(str).str.contains('burglary', case=False, na=False).sum()),
          theft_count=('crime', lambda x: x.astype(str).str.lower().eq('theft').sum()),
          vehicle_count=('crime', lambda x: x.astype(str).str.contains('vehicle', case=False, na=False).sum()),
          robbery_count=('crime', lambda x: x.astype(str).str.contains('robbery', case=False, na=False).sum()),
          drugs_count=('crime', lambda x: x.astype(str).str.contains('drugs', case=False, na=False).sum()),
          stabbing_count=('crime', lambda x: x.astype(str).str.contains('stabbing', case=False, na=False).sum()),

          avg_victim_age=('victim_age', 'mean'),
          female_victim_ratio=('sex', lambda x: x.astype(str).str.lower().eq('f').mean()),
          urban_ratio=('land_use_type', lambda x: x.astype(str).eq('General Urban').mean()),

          population=('gn_population', 'first'),
          mean_distance_m=('gn_distance_m', 'mean'),

          holiday_ratio=('is_holiday', lambda x: pd.to_numeric(x, errors='coerce').mean()),
          rainy_ratio=('weather', lambda x: x.astype(str).str.lower().eq('rainy').mean()),

          ds_division=('DS_Division', 'first')
      )
      .reset_index()
)

##feature engineering and future-safe lagged features

In [ ]:
## Create complete GN x Week grid

all_gn = aggregated_df['gn_division'].dropna().unique()

all_weeks = pd.date_range(
    aggregated_df['week'].min(),
    aggregated_df['week'].max(),
    freq='W-MON'
)

full_index = pd.MultiIndex.from_product(
    [all_gn, all_weeks],
    names=['gn_division', 'week']
)

completed_df = (
    full_index.to_frame(index=False)
    .merge(aggregated_df, on=['gn_division', 'week'], how='left')
)

crime_cols = [
    'total_crimes', 'burglary_count', 'theft_count',
    'vehicle_count', 'robbery_count', 'drugs_count', 'stabbing_count'
]

completed_df[crime_cols] = completed_df[crime_cols].fillna(0)

completed_df['avg_victim_age'] = (
    completed_df.groupby('gn_division')['avg_victim_age']
    .transform(lambda x: x.fillna(x.median()))
)

ratio_cols = ['female_victim_ratio', 'urban_ratio', 'holiday_ratio', 'rainy_ratio']
completed_df[ratio_cols] = completed_df[ratio_cols].fillna(0)

completed_df[['population', 'mean_distance_m']] = (
    completed_df.groupby('gn_division')[['population', 'mean_distance_m']]
    .ffill()
    .bfill()
)

In [ ]:
## Integrate demographic & environmental variables

# Merge static GN context

context_cols = [
    'Unemployment_Rate',
    'Building_Density',
    'Avg_Household_Income',
    'Land_Area_Density',
    'Road_Density'
]

for col in context_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

gn_context = (
    df.groupby('gn_division')
      .agg(
          unemployment_rate=('Unemployment_Rate', 'mean'),
          avg_income=('Avg_Household_Income', 'mean'),
          building_density=('Building_Density', 'mean'),
          land_area_density=('Land_Area_Density', 'mean'),
          road_density=('Road_Density', 'mean'),
          ds_division=('DS_Division', 'first')
      )
      .reset_index()
)

completed_df = completed_df.merge(
    gn_context,
    on='gn_division',
    how='left'
)


# Feature engineering

## Sort before temporal feature creation
df_model = completed_df.copy()
df_model = df_model.sort_values(['gn_division', 'week']).reset_index(drop=True)

static_feature_cols = [
    'unemployment_rate',
    'avg_income',
    'building_density',
    'land_area_density',
    'road_density'
]

for col in static_feature_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())



## Time features
df_model['year'] = df_model['week'].dt.year
df_model['week_of_year'] = df_model['week'].dt.isocalendar().week.astype(int)
df_model['week_sin'] = np.sin(2 * np.pi * df_model['week_of_year'] / 52)
df_model['week_cos'] = np.cos(2 * np.pi * df_model['week_of_year'] / 52)

## Scale-friendly transforms
df_model['log_population'] = np.log1p(df_model['population'])
df_model['distance_km'] = df_model['mean_distance_m'] / 1000

## Create next-week targets
crime_types = {
    'burglary': 'y_burglary',
    'theft': 'y_theft',
    'vehicle': 'y_vehicle',
    'robbery': 'y_robbery',
    'drugs': 'y_drugs',
    'stabbing': 'y_stabbing'
}

for crime in crime_types.keys():
    df_model[f'y_{crime}'] = (
        df_model.groupby('gn_division')[f'{crime}_count']
        .shift(-1)
        .fillna(0)
        .gt(0)
        .astype(int)
    )

# Future-safe lagged versions of weekly incident-derived contextual variables
incident_context_cols = [
    'avg_victim_age',
    'female_victim_ratio',
    'urban_ratio',
    'holiday_ratio',
    'rainy_ratio'
]

for col in incident_context_cols:
    df_model[f'{col}_lag1'] = df_model.groupby('gn_division')[col].shift(1)
    df_model[f'{col}_lag2'] = df_model.groupby('gn_division')[col].shift(2)
    df_model[f'{col}_roll4'] = (
        df_model.groupby('gn_division')[col]
        .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
    )

# Crime-history lag and rolling features
crime_count_cols = [
    'total_crimes', 'burglary_count', 'theft_count',
    'vehicle_count', 'robbery_count', 'drugs_count', 'stabbing_count'
]

for col in crime_count_cols:
    df_model[f'{col}_lag1'] = df_model.groupby('gn_division')[col].shift(1)
    df_model[f'{col}_lag2'] = df_model.groupby('gn_division')[col].shift(2)
    df_model[f'{col}_lag4'] = df_model.groupby('gn_division')[col].shift(4)

    df_model[f'{col}_roll4'] = (
        df_model.groupby('gn_division')[col]
        .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
    )

    df_model[f'{col}_roll8'] = (
        df_model.groupby('gn_division')[col]
        .transform(lambda s: s.shift(1).rolling(window=8, min_periods=1).mean())
    )
    df_model[f'{col}_trend_4v8'] = df_model[f'{col}_roll4'] - df_model[f'{col}_roll8']


# Fill lagged columns
feature_fill_cols = [
    c for c in df_model.columns
    if ('_lag' in c or '_roll' in c or '_trend_' in c)
]
df_model[feature_fill_cols] = df_model[feature_fill_cols].fillna(0)

# Keep counts as int
df_model[crime_cols] = df_model[crime_cols].astype(int)

# drop last week of each GN because shift(-1) created synthetic zeros
last_week_mask = df_model.groupby('gn_division')['week'].transform('max') == df_model['week']
df_model = df_model.loc[~last_week_mask].copy()

# cleaning duplicated columns
df_model.rename(columns={'ds_division_y': 'ds_division'}, inplace=True, errors='ignore')
df_model.drop(columns=['ds_division_x'], inplace=True, errors='ignore')

In [ ]:
# saving the preprocessed dataset
df_model.to_csv("df_model_engineered.csv", index=False)
files.download("df_model_engineered.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Base features shared by all crime models
BASE_FEATURES = [
    'unemployment_rate',
    'avg_income',
    'building_density',
    'land_area_density',
    'road_density',
    'log_population',
    'distance_km',
    'year',
    'week_of_year',
    'week_sin',
    'week_cos',

    # future-safe lagged contextual variables
    'avg_victim_age_lag1',
    'avg_victim_age_lag2',
    'avg_victim_age_roll4',

    'female_victim_ratio_lag1',
    'female_victim_ratio_lag2',
    'female_victim_ratio_roll4',

    'urban_ratio_lag1',
    'urban_ratio_lag2',
    'urban_ratio_roll4',

    'holiday_ratio_lag1',
    'holiday_ratio_lag2',
    'holiday_ratio_roll4',

    'rainy_ratio_lag1',
    'rainy_ratio_lag2',
    'rainy_ratio_roll4',

    # general crime pressure
    'total_crimes_lag1',
    'total_crimes_lag2',
    'total_crimes_lag4',
    'total_crimes_roll4',
    'total_crimes_roll8',
    'total_crimes_trend_4v8'
]


CRIME_SPECIFIC_FEATURES = {
    'burglary': BASE_FEATURES + [
        'burglary_count_lag1',
        'burglary_count_lag2',
        'burglary_count_lag4',
        'burglary_count_roll4',
        'burglary_count_roll8',
        'burglary_count_trend_4v8'
    ],
    'theft': BASE_FEATURES + [
        'theft_count_lag1',
        'theft_count_lag2',
        'theft_count_lag4',
        'theft_count_roll4',
        'theft_count_roll8',
        'theft_count_trend_4v8'
    ],
    'vehicle': BASE_FEATURES + [
        'vehicle_count_lag1',
        'vehicle_count_lag2',
        'vehicle_count_lag4',
        'vehicle_count_roll4',
        'vehicle_count_roll8',
        'vehicle_count_trend_4v8'
    ],
    'robbery': BASE_FEATURES + [
        'robbery_count_lag1',
        'robbery_count_lag2',
        'robbery_count_lag4',
        'robbery_count_roll4',
        'robbery_count_roll8',
        'robbery_count_trend_4v8'
    ],
    'drugs': BASE_FEATURES + [
        'drugs_count_lag1',
        'drugs_count_lag2',
        'drugs_count_lag4',
        'drugs_count_roll4',
        'drugs_count_roll8',
        'drugs_count_trend_4v8'
    ],
    'stabbing': BASE_FEATURES + [
        'stabbing_count_lag1',
        'stabbing_count_lag2',
        'stabbing_count_lag4',
        'stabbing_count_roll4',
        'stabbing_count_roll8',
        'stabbing_count_trend_4v8'
    ]
}

In [ ]:
# Helper Functions
def low_support_warning(y_train, y_test, crime_name, min_test_positives=20):
    train_pos = int(y_train.sum())
    test_pos = int(y_test.sum())

    print(f"\n[{crime_name.upper()}] Positive support -> train: {train_pos}, test: {test_pos}")

    if test_pos < min_test_positives:
        print(f"WARNING: {crime_name} has very low positive support in test set.")
        print("Interpret classification metrics cautiously; this class is rare and unstable.")


def apply_smote(X_train, y_train, random_state=42):
    minority_count = int(y_train.sum())

    if minority_count < 6:
        print(f"      Skipping SMOTE: only {minority_count} minority samples.")
        return X_train, y_train

    k = min(5, minority_count - 1)
    sm = SMOTE(k_neighbors=k, random_state=random_state)
    X_res, y_res = sm.fit_resample(X_train, y_train)

    print(f"      SMOTE applied  →  class 0: {(y_res == 0).sum()}  class 1: {(y_res == 1).sum()}")
    return X_res, y_res


def find_best_threshold(model, X_val, y_val):
    if y_val.sum() == 0:
        return 0.5

    proba = model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, proba)

    f1s = np.where(
        (precisions + recalls) == 0,
        0,
        2 * precisions * recalls / (precisions + recalls)
    )

    best_idx = np.argmax(f1s)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    print(
        f"      Best threshold: {best_thr:.3f}  "
        f"(val F1={f1s[best_idx]:.3f}  "
        f"prec={precisions[best_idx]:.3f}  "
        f"rec={recalls[best_idx]:.3f})"
    )

    return float(best_thr)

# Model Benchmarking

In [ ]:
def benchmarking(X_train_scaled, X_test_scaled, y_train, y_test, crime_name):
    pos = y_train.sum()
    neg = len(y_train) - pos
    scale_weight = neg / pos if pos > 0 else 1

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_weight,
            eval_metric='logloss',
            random_state=42
        )
    }

    benchmark_results = []

    for name, model in models.items():
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        benchmark_results.append({
            "Model": name,
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            "PR-AUC": average_precision_score(y_test, y_proba),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1-score": f1_score(y_test, y_pred, zero_division=0)
        })

    benchmark_df = pd.DataFrame(benchmark_results).sort_values(by="PR-AUC", ascending=False)

    print(f"\n{'='*70}")
    print(f"  BENCHMARK — {crime_name.upper()} (default threshold)")
    print(f"{'='*70}")
    print(benchmark_df.to_string(index=False))

    return benchmark_df


#Hyperparameter Tuning

In [ ]:
def expanding_window_splits(n_samples, initial_train_size=0.5, val_size=0.15, n_splits=3):
    """
    Create expanding-window train/validation splits.

    Example:
    Fold 1: first 50% train, next 15% validate
    Fold 2: first 65% train, next 15% validate
    Fold 3: first 80% train, next 15% validate

    All sizes are proportions of the provided training set.
    """
    splits = []

    init_end = int(n_samples * initial_train_size)
    val_len = int(n_samples * val_size)

    for i in range(n_splits):
        train_end = init_end + i * val_len
        val_end = train_end + val_len

        if val_end > n_samples:
            break

        train_idx = np.arange(0, train_end)
        val_idx = np.arange(train_end, val_end)

        if len(train_idx) > 0 and len(val_idx) > 0:
            splits.append((train_idx, val_idx))

    return splits

In [ ]:
def hyperparameter_tuning(X_train_scaled, y_train, crime_name=""):
    """
    Hyperparameter tuning using expanding-window validation.
    Selects params based on average validation F1 across folds.
    Also reports average PR-AUC for extra insight.
    """
    splits = expanding_window_splits(
        n_samples=len(X_train_scaled),
        initial_train_size=0.5,
        val_size=0.15,
        n_splits=3
    )

    param_grid = {
        'n_estimators': [200, 400],
        'max_depth': [None, 10, 20],
        'min_samples_leaf': [1, 3, 5],
        'max_features': ['sqrt', 0.5]
    }

    best_score = -1
    best_params = None
    best_thr = 0.5

    all_results = []

    for params in itertools.product(
        param_grid['n_estimators'],
        param_grid['max_depth'],
        param_grid['min_samples_leaf'],
        param_grid['max_features']
    ):
        n_est, max_d, min_leaf, max_feat = params

        fold_f1s = []
        fold_pr_aucs = []
        fold_thresholds = []

        for fold_num, (train_idx, val_idx) in enumerate(splits, start=1):
            X_tr_sub = X_train_scaled.iloc[train_idx]
            y_tr_sub = y_train.iloc[train_idx]

            X_val = X_train_scaled.iloc[val_idx]
            y_val = y_train.iloc[val_idx]

            # skip fold if validation has no positives
            if y_val.sum() == 0:
                print(f"      Fold {fold_num}: skipped (no positive samples in validation)")
                continue

            # apply SMOTE only on training portion of fold
            X_tr_res, y_tr_res = apply_smote(X_tr_sub, y_tr_sub)

            rf = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=max_d,
                min_samples_leaf=min_leaf,
                max_features=max_feat,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )

            rf.fit(X_tr_res, y_tr_res)

            proba_val = rf.predict_proba(X_val)[:, 1]
            thr = find_best_threshold(rf, X_val, y_val)
            y_pred_val = (proba_val >= thr).astype(int)

            val_f1 = f1_score(y_val, y_pred_val, zero_division=0)
            val_pr_auc = average_precision_score(y_val, proba_val)

            fold_f1s.append(val_f1)
            fold_pr_aucs.append(val_pr_auc)
            fold_thresholds.append(thr)

        if len(fold_f1s) == 0:
            continue

        mean_f1 = np.mean(fold_f1s)
        mean_pr_auc = np.mean(fold_pr_aucs)
        mean_thr = float(np.mean(fold_thresholds))

        all_results.append({
            'n_estimators': n_est,
            'max_depth': max_d,
            'min_samples_leaf': min_leaf,
            'max_features': max_feat,
            'mean_f1': mean_f1,
            'mean_pr_auc': mean_pr_auc,
            'mean_threshold': mean_thr
        })

        if mean_f1 > best_score:
            best_score = mean_f1
            best_params = {
                'n_estimators': n_est,
                'max_depth': max_d,
                'min_samples_leaf': min_leaf,
                'max_features': max_feat
            }
            best_thr = mean_thr

    if len(all_results) == 0:
        print(f"\nNo valid tuning folds found for {crime_name}.")
        return None, 0.5, pd.DataFrame()

    results_df = pd.DataFrame(all_results).sort_values(
        by=['mean_f1', 'mean_pr_auc'],
        ascending=False
    )

    print(f"\nTop tuning results for {crime_name}:")
    print(results_df.head(10).to_string(index=False))

    print(
        f"\n[{crime_name}] Best params: {best_params}  "
        f"mean CV-F1={best_score:.3f}  mean threshold={best_thr:.3f}"
    )

    return best_params, best_thr, results_df

In [ ]:

def get_global_summary_table(shap_values_pos, feature_names):
    mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
    return (
        pd.DataFrame({
            'feature': feature_names,
            'mean_abs_shap': mean_abs_shap
        })
        .sort_values(by='mean_abs_shap', ascending=False)
    )


def get_global_directional_effect_table(shap_values_pos, X):
    effects = {}

    for i, col in enumerate(X.columns):
        if X[col].nunique() <= 1:
            effects[col] = 'direction unclear'
            continue

        corr = np.corrcoef(X[col], shap_values_pos[:, i])[0, 1]

        if np.isnan(corr):
            effects[col] = 'direction unclear'
        else:
            effects[col] = 'increases risk' if corr > 0 else 'decreases risk'

    return effects


def get_local_top_drivers(shap_row, X_row, top_k=3):
    contribs = pd.DataFrame({
        'feature': X_row.index,
        'shap_value': shap_row,
        'feature_value': X_row.values
    })

    return contribs.reindex(
        contribs.shap_value.abs().sort_values(ascending=False).index
    ).head(top_k)


def local_explanation(crime_type, gn_division, top_drivers):
    drivers = ", ".join(top_drivers['feature'].values[:2])
    mitigator = top_drivers['feature'].values[-1]

    return (
        f"The elevated {crime_type} risk in {gn_division} during the past week "
        f"is primarily driven by {drivers}, which outweighs mitigating factors "
        f"such as {mitigator}."
    )

def shap_crime(crime_name, trained_model, df_model, X_test_scaled, test_index):
    print(f"Causative Factors for {crime_name}")

    rf_model = trained_model
    y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_scaled)

    if isinstance(shap_values, list):
        shap_values_pos = shap_values[1]
    elif len(np.array(shap_values).shape) == 3:
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    global_summary = get_global_summary_table(
        shap_values_pos,
        X_test_scaled.columns
    )

    directional_effect = get_global_directional_effect_table(
        shap_values_pos,
        X_test_scaled
    )

    local_results = []

    for i in range(len(X_test_scaled)):
        top_drivers = get_local_top_drivers(
            shap_values_pos[i],
            X_test_scaled.iloc[i]
        )

        local_results.append({
            'gn_division': df_model.loc[test_index[i], 'gn_division'],
            'week': str(df_model.loc[test_index[i], 'week']),
            'risk_score': float(y_pred_proba_rf[i]),
            'top_features': list(top_drivers['feature']),
            'explanation_text': local_explanation(
                crime_name,
                df_model.loc[test_index[i], 'gn_division'],
                top_drivers
            )
        })

    return {
        'model': rf_model,
        'global_summary': global_summary,
        'directional_effect': directional_effect,
        'local_results': local_results
    }


#Train and Evaluate the Model

In [ ]:
#Feature existence check before training

for crime_name, feats in CRIME_SPECIFIC_FEATURES.items():
    missing = [f for f in feats if f not in df_model.columns]
    if missing:
        raise ValueError(f"Missing features for {crime_name}: {missing}")


In [ ]:
results = {}
thresholds = {}
feature_lists = {}
metrics_summary = []

split_index = int(len(df_model) * 0.8)

for crime_name, target_col in crime_types.items():
    print(f"\n\n{'#'*80}")
    print(f"PROCESSING CRIME TYPE: {crime_name.upper()}")
    print(f"{'#'*80}")

    selected_features = CRIME_SPECIFIC_FEATURES[crime_name]
    feature_lists[crime_name] = selected_features

    X = df_model[selected_features]
    y = df_model[target_col]

    X_train = X.iloc[:split_index].copy()
    X_test = X.iloc[split_index:].copy()

    y_train = y.iloc[:split_index].copy()
    y_test = y.iloc[split_index:].copy()

    low_support_warning(y_train, y_test, crime_name)

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    benchmark_df = benchmarking(X_train_scaled, X_test_scaled, y_train, y_test, crime_name)

    best_params, best_thr, tuning_df = hyperparameter_tuning(
        X_train_scaled,
        y_train,
        crime_name
    )

    if best_params is None:
        best_params = {
            'n_estimators': 300,
            'max_depth': None,
            'min_samples_leaf': 3,
            'max_features': 'sqrt'
        }
        best_thr = 0.5

    rf_tuned = RandomForestClassifier(
        **best_params,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    X_train_final, y_train_final = apply_smote(X_train_scaled, y_train)
    rf_tuned.fit(X_train_final, y_train_final)

    proba_test = rf_tuned.predict_proba(X_test_scaled)[:, 1]
    y_pred_tuned = (proba_test >= best_thr).astype(int)

    print(f"\nPR-AUC for {crime_name}: {average_precision_score(y_test, proba_test):.4f}")
    print(f"ROC-AUC for {crime_name}: {roc_auc_score(y_test, proba_test):.4f}")
    print(f"\nClassification Report for {crime_name}")
    print(f"Threshold used: {best_thr:.3f}")
    print(classification_report(y_test, y_pred_tuned, zero_division=0))

    thresholds[crime_name] = best_thr

    shap_bundle = shap_crime(
        crime_name,
        rf_tuned,
        df_model,
        X_test_scaled,
        X_test.index
    )

    results[crime_name] = {
        'model': rf_tuned,
        'scaler': scaler,
        'features': selected_features,
        'threshold': best_thr,
        'benchmark_df': benchmark_df,
        'tuning_df': tuning_df,
        'shap_bundle': shap_bundle
    }

    print(f"\nTop global drivers for {crime_name}:")
    print(shap_bundle['global_summary'].head(10).to_string(index=False))

    final_metrics = {
        'crime': crime_name,
        'roc_auc': roc_auc_score(y_test, proba_test),
        'pr_auc': average_precision_score(y_test, proba_test),
        'precision': precision_score(y_test, y_pred_tuned, zero_division=0),
        'recall': recall_score(y_test, y_pred_tuned, zero_division=0),
        'f1': f1_score(y_test, y_pred_tuned, zero_division=0),
        'threshold': best_thr,
        'test_support_positive': int(y_test.sum())
    }

    metrics_summary.append(final_metrics)






################################################################################
PROCESSING CRIME TYPE: BURGLARY
################################################################################

[BURGLARY] Positive support -> train: 375, test: 45

  BENCHMARK — BURGLARY (default threshold)
              Model  ROC-AUC   PR-AUC  Precision   Recall  F1-score
            XGBoost 0.733587 0.036264   0.076923 0.022222  0.034483
Logistic Regression 0.718231 0.025381   0.022444 0.200000  0.040359
      Random Forest 0.652181 0.018080   0.000000 0.000000  0.000000
  Gradient Boosting 0.623711 0.014997   0.000000 0.000000  0.000000
      SMOTE applied  →  class 0: 8269  class 1: 8269
      Best threshold: 0.520  (val F1=0.178  prec=0.178  rec=0.178)
      SMOTE applied  →  class 0: 10743  class 1: 10743
      Best threshold: 0.685  (val F1=0.062  prec=0.077  rec=0.053)
      SMOTE applied  →  class 0: 13271  class 1: 13271
      Best threshold: 0.410  (val F1=0.025  prec=0.013  rec=0.263)
   

# Save Artifacts

In [ ]:
MODEL_DIR = "models"

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)
    print(f"{MODEL_DIR} created")
else:
    print(f"{MODEL_DIR} already exists")

pd.DataFrame(metrics_summary).to_csv(f"{MODEL_DIR}/final_metrics_summary.csv", index=False)


for crime, result in results.items():
    joblib.dump(result['model'], f"{MODEL_DIR}/rf_{crime}.pkl")
    joblib.dump(result['scaler'], f"{MODEL_DIR}/scaler_{crime}.pkl")

    result['benchmark_df'].to_csv(f"{MODEL_DIR}/benchmark_{crime}.csv", index=False)
    result['tuning_df'].to_csv(f"{MODEL_DIR}/tuning_{crime}.csv", index=False)

    result['shap_bundle']['global_summary'].to_csv(
        f"{MODEL_DIR}/global_summary_{crime}.csv",
        index=False
    )

    pd.DataFrame.from_dict(
        result['shap_bundle']['directional_effect'],
        orient='index',
        columns=['effect']
    ).reset_index().rename(columns={'index': 'feature'}).to_csv(
        f"{MODEL_DIR}/directional_effect_{crime}.csv",
        index=False
    )

    pd.DataFrame(result['shap_bundle']['local_results']).to_csv(
        f"{MODEL_DIR}/local_results_{crime}.csv",
        index=False
    )

with open(f"{MODEL_DIR}/feature_lists.json", "w") as f:
    json.dump(feature_lists, f, indent=4)

with open(f"{MODEL_DIR}/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=4)

metadata = {
    "crime_types": list(crime_types.keys()),
    "model_type": "RandomForest",
    "time_granularity": "weekly",
    "unit": "GN Division",
    "prediction_target": "next-week crime occurrence by GN division",
    "note": "Rare classes such as vehicle and stabbing may have unstable metrics due to low positive support."
}

with open(f"{MODEL_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

df_model.to_csv(f"{MODEL_DIR}/df_model_engineered_final.csv", index=False)

print(os.getcwd())



models created
/content


In [ ]:
import shutil


print("\nSaved files in models/:")
for file in sorted(os.listdir(MODEL_DIR)):
    print("-", file)

archive_path = shutil.make_archive("models_bundle", "zip", MODEL_DIR)

print(f"\nCreated archive: {archive_path}")

files.download(archive_path)


Saved files in models/:
- benchmark_burglary.csv
- benchmark_drugs.csv
- benchmark_robbery.csv
- benchmark_stabbing.csv
- benchmark_theft.csv
- benchmark_vehicle.csv
- df_model_engineered_final.csv
- directional_effect_burglary.csv
- directional_effect_drugs.csv
- directional_effect_robbery.csv
- directional_effect_stabbing.csv
- directional_effect_theft.csv
- directional_effect_vehicle.csv
- feature_lists.json
- final_metrics_summary.csv
- global_summary_burglary.csv
- global_summary_drugs.csv
- global_summary_robbery.csv
- global_summary_stabbing.csv
- global_summary_theft.csv
- global_summary_vehicle.csv
- local_results_burglary.csv
- local_results_drugs.csv
- local_results_robbery.csv
- local_results_stabbing.csv
- local_results_theft.csv
- local_results_vehicle.csv
- metadata.json
- rf_burglary.pkl
- rf_drugs.pkl
- rf_robbery.pkl
- rf_stabbing.pkl
- rf_theft.pkl
- rf_vehicle.pkl
- scaler_burglary.pkl
- scaler_drugs.pkl
- scaler_robbery.pkl
- scaler_stabbing.pkl
- scaler_theft.pkl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>